In [4]:
#HEy buddy here we practice the yt project right okey so let start
# So hame yt ki jo vidoes hai usme questino ke answer dene wala bot banana hai right so first ham kya kerenge ham api key ko insert kerenge
# jiski vajah se hamara jo model hai output de sake right so let insert it
import os
os.environ["OPENAI_API_KEY"] ="gsk_MEXI2PhbLZMhXOr8luqWOmTrY7m0y"  # so maine ye api key trim ki hui hai to yaha pe jo bhi banda use kerna chahta ho vo apni api key dale
# To finally hamne apni key ko insert ker liya ab next step me jo important updates hai unhe
# downlaod ker lenge right


In [5]:
# now import the librarires
!pip install -q youtube-transcript-api langchain-community langchain-openai \
               faiss-cpu tiktoken python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [7]:
# now ab jo bhi important libraries hai hame vo import kerni padegi right so lets do it okey
from youtube_transcript_api import YouTubeTranscriptApi,TranscriptsDisabled # ye wali lib to isliye install ki kunki hame yt ki transcript chaiiye right and agar nahi hai to ham disabled ka use kerke message show ker sakte hai ki bhai nahi hai ok
from langchain_text_splitters import RecursiveCharacterTextSplitter # ab text splitter isiliye kunki age jake hame chunks me slit kerne padenge to usi vajah se hame isski jarroat padi right
from langchain_openai import ChatOpenAI # ham model chatOpenAI use kerenge but ise age jake ham iske ander grok ka model use kerenge just name hamne ChatOpenAI diya hai kunki grok name ka koi  bhi variable database me nahi hai ok
from langchain_community.vectorstores import FAISS # ye isiliye kunki hamne FAiss kause kereke apna conteext ko store bhi to kerne hai
from langchain_core.prompts import PromptTemplate # iska to pata hi ki hamara jo formate hoga output vo ham isme bata denge ki kaise kerna hai taki uske bad ye vaisa format ehi sue kere output ka right
import os
os.environ["OPENAI_API_KEY"] ="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"
llm = ChatOpenAI(
    model="llama-3.1-8b-instant",
    openai_api_key = os.environ["OPENAI_API_KEY"],
    openai_api_base="https://api.groq.com/openai/v1"
)
# to yaha pe sab important libraries downlaod ho chuki hai sucessfully to sabse first step pe ate hai hamrae rag me hamne 4 steps padhe the na
# indexing
# Retrival
# Augumentation
# Generation



In [11]:
# Ist one is indexing
try:
  transcript_list = YouTubeTranscriptApi().fetch("ldxFjLJ3rVY",languages=["en"])
  transcript =  " ".join(chunks.text for chunks in transcript_list)
  print(transcript)
except TranscriptsDisabled:
  print("No captions avilable on this video ")
# summary ye hai ki transcript list ko nikalte time hame yt ki api ki jarroat hoti hai and use hamne fetch kerliya to fetch kerte time 2 parameter dete hai ek to jis video ki transcript nikalni hai uski id and second languages
# ab iske bad iske pass kafi chunks aajate hai ab use  join kerek ek string me long strring me convert kerna hai to vo kerta hai hamara join funtion and iske ander hamne ek loop laga di taki sabhi pe repeatidly same kam hota rahe right kunki isme text jo row text hota hai vo alga alag chote chunks me already hota hai bus use jouin kerna hota hai ok
# then print ker diya ab agar us video pe transcript ho hi na to hamne disabled ka tag laga diya and kehh diya no caption print ker do right

Whenever I'm making one of these videos, there's sometimes a special moment where the act of animating involves solving a whole bunch of little technical puzzlers, and then the underlying math I'm trying to explain clicks for me in a way that it hadn't before once I see it alive on screen. The best versions of those moments often tell me when a video is going to be one of my favorites, and putting together the end of this piece right here was one such time when I got that feeling. Our story doesn't actually start in the math classroom today. We begin in the art room. Imagine standing in a gallery, looking at a picture of a boat in a harbour, and the whole world warps as your gaze shifts upwards and to the right, where you see a village on this waterfront. Among the tightly clustered buildings, the world warps even more as your gaze shifts downwards to the entrance of one building, leading to a hallway full of artwork. And at the end of this hall, here you are again, staring at a pictur

In [21]:
# Now the work of text splitter is there so let start
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=200)
chunks=splitter.create_documents([transcript])
#sumary kya ha ki hamne ek spiltter ka use kiya chote chote chunks me divide kerne ke liye
# ab ek chunk me max 1000 characters allowed hai and overlap hamne 200 ka kiya taki agar context beech me se break bhi ho jaye
# to next line me vo dubara se start ho jaye okey then hamne un sabhi chunks ko ek document em create ker ke convert ker diya
#for doc in chunks:
  #print(doc.page_content)
  # And finally ham print ker ke dekh sakte hain ki hamrae jo ch
print(len(transcript))
for i, doc in enumerate(chunks):
    print(f"Chunk {i}:\n{doc.page_content}\n")

47043
Chunk 0:
Whenever I'm making one of these videos, there's sometimes a special moment where the act of animating involves solving a whole bunch of little technical puzzlers, and then the underlying math I'm trying to explain clicks for me in a way that it hadn't before once I see it alive on screen. The best versions of those moments often tell me when a video is going to be one of my favorites, and putting together the end of this piece right here was one such time when I got that feeling. Our story

Chunk 1:
versions of those moments often tell me when a video is going to be one of my favorites, and putting together the end of this piece right here was one such time when I got that feeling. Our story doesn't actually start in the math classroom today. We begin in the art room. Imagine standing in a gallery, looking at a picture of a boat in a harbour, and the whole world warps as your gaze shifts upwards and to the right, where you see a village on this waterfront. Among the tig

In [25]:
# ABHI TAK HAMNE TRANSCRIPT NIKALI CHUNKS ME DIVIDE KERLIYA AB HAR CHUNKS KA VECTOR BANAYENGE RIGHT KUNKI TABHI HAM COMPARRISON
# KER SAKTE HAIN NA AND TOP 3 YA 4 RESULT NIKAL KE DE SAKTE HAIN BAD ME UNHE COMBINE KERKE USER KI QUERY KA ANSWER HOGA RIGHT
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
embeddings=HuggingFaceEmbeddings()
vector_store=FAISS.from_documents(chunks,embeddings)
#summary isme hamarae vector me conert ho rhe hain and then unhe ek vector store me dal rhe hai store ker rhe hai taki bad me unpe search veghera ki ja sake and similairatiy search perform ker sake


/tmp/ipykernel_627/1010700658.py:5: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings=HuggingFaceEmbeddings()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [27]:
# ab ham chahe ge ki har ek chunks ki apni ek id bane to
vector_store.index_to_docstore_id

{0: '16f75c7a-83c6-4371-ab1c-6d93cdaf0294',
 1: 'c683d353-f0ac-4761-94c3-cccd278c4b95',
 2: '80ed0eb2-a274-4c6b-8ccb-0d387b6ed356',
 3: '0995f427-2776-4c05-bc30-208ba1fc05f4',
 4: '43702624-e0d4-4296-8574-93c02ee32279',
 5: '14fd0704-1128-45aa-a5a5-200fbdec0fc6',
 6: '1732ec58-a2cb-4659-8216-b1993de7f4c5',
 7: '6a390de5-aee6-44d0-ae94-6e8a15db13d4',
 8: '19568b29-79b4-417d-930f-f74d82ab12f6',
 9: '8969b58a-23ab-4533-95e4-ec9be21b7899',
 10: '49421f2f-8f6d-4bfb-b68e-a331a041a74a',
 11: 'd1525e1a-8ed0-4564-a899-50ec76f3223b',
 12: '26dafdb3-fa24-49ad-9408-0e2f699d6e26',
 13: 'e8e13746-fcc5-44fa-bc2c-12fdca8c37fc',
 14: 'c1823e37-39e3-42b4-bbe7-93771b66bc29',
 15: '14fe5505-e631-4032-bbb4-b7496465a8fa',
 16: 'b7da41a5-e9f2-4d18-a4a5-a3c5e5f18a4c',
 17: '7ac42046-a49d-4aae-9f94-f556e878ecf3',
 18: 'f27c3dbc-c98a-41e5-9b4b-b29872a74e30',
 19: 'a5e6a0ad-c82a-4073-a34f-cf47baf6f39d',
 20: 'd11ef7b5-dade-4c29-98fa-2945904ef2b5',
 21: '9c9f1b54-d140-4cbe-b89a-8c3772168b60',
 22: 'f550825d-3b33-

In [35]:
# ab kam ayega retreiver ka right and similarity search perform kerna ok
retriver = vector_store.as_retriever(search_type="similarity",search_kwargs={"k":3})
retriver.invoke("How (and why) to take a logarithm of an image")

[Document(id='6a15f0ad-2398-42ac-a908-02c62acf59af', metadata={}, page_content='say the example we were working with earlier with the pi creature looking at a picture of a house where that same pi creature lives. In other words, it is finally time for you and me to answer that question of what it means to take the natural log of a picture. Okay, so think about this for a second. You already know that a vertical line segment like this one on the left with a height of 2 pi gets turned into a circle when you apply the map e to the z. So the natural log is going to take a'),
 Document(id='139a0b9c-6a72-4ce9-a10e-ce9b42684e5c', metadata={}, page_content="nice as to think of the log as a multi-valued function, where each point on the right corresponds to a repeating sequence of points on the left, spaced out two pi vertically, going infinitely in both directions. Now in our special case, you also see this repeating pattern as you move to the left, but that is something different entirely. Yo

In [36]:
#3 part is augumentation
# so no next part hai 3rd part vo hai humara augumentaiton right so lets do it
prompt = PromptTemplate(
    template="""
    You are a very helpful assistant
    Answer Only from the provided transcript context.
    If the context is insufficent , just say that you dont know.

    IMPORTANT:
    - Always answer in Hindi (simple and clear)
    {context}
    Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [41]:
question="How (and why) to take a logarithm of an image"
retriever = vector_store.as_retriever()
retrieved_docs = retriever.invoke(question)
for doc in retrieved_docs:
    print(doc.page_content)

say the example we were working with earlier with the pi creature looking at a picture of a house where that same pi creature lives. In other words, it is finally time for you and me to answer that question of what it means to take the natural log of a picture. Okay, so think about this for a second. You already know that a vertical line segment like this one on the left with a height of 2 pi gets turned into a circle when you apply the map e to the z. So the natural log is going to take a
nice as to think of the log as a multi-valued function, where each point on the right corresponds to a repeating sequence of points on the left, spaced out two pi vertically, going infinitely in both directions. Now in our special case, you also see this repeating pattern as you move to the left, but that is something different entirely. You would not see this for most images. It arises specifically because we're working with a self-similar Drosta image, one that looks identical as you zoom in
and th

In [43]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"say the example we were working with earlier with the pi creature looking at a picture of a house where that same pi creature lives. In other words, it is finally time for you and me to answer that question of what it means to take the natural log of a picture. Okay, so think about this for a second. You already know that a vertical line segment like this one on the left with a height of 2 pi gets turned into a circle when you apply the map e to the z. So the natural log is going to take a\n\nnice as to think of the log as a multi-valued function, where each point on the right corresponds to a repeating sequence of points on the left, spaced out two pi vertically, going infinitely in both directions. Now in our special case, you also see this repeating pattern as you move to the left, but that is something different entirely. You would not see this for most images. It arises specifically because we're working with a self-similar Drosta image, one that looks identical as you zoom in\n\

In [45]:
final_prompt = prompt.invoke({"context":context_text,"question":question})
final_prompt

StringPromptValue(text="\n    You are a very helpful assistant\n    Answer Only from the provided transcript context.\n    If the context is insufficent , just say that you dont know.\n\n    IMPORTANT:\n    - Always answer in Hindi (simple and clear)\n    say the example we were working with earlier with the pi creature looking at a picture of a house where that same pi creature lives. In other words, it is finally time for you and me to answer that question of what it means to take the natural log of a picture. Okay, so think about this for a second. You already know that a vertical line segment like this one on the left with a height of 2 pi gets turned into a circle when you apply the map e to the z. So the natural log is going to take a\n\nnice as to think of the log as a multi-valued function, where each point on the right corresponds to a repeating sequence of points on the left, spaced out two pi vertically, going infinitely in both directions. Now in our special case, you also 

In [48]:
answer = llm.invoke(final_prompt)
print(answer.content)

Logaare ke liye, hum aise sochte hai: phir is image ko kaalpanik jagah par le jaein aur use e ke sath jod lein. Isse humko e^z ke map ban jaega. Ab, logaare ke liye hume e^z ki inverted function ki zaroorat hoti hai.

Iske liye, hum aise sochte hai: agar hum vertical line segment ke height 2*pi hai toh usse circle ban jaega, jiska radius 1 hai. Iske baad, hum y coordinate ka log le sakte hai, jiska matlab hoga ki hum y coordinate ko -1 aur pi tak rakheinge. Isse hume y^2+1=0 ka equation milta hai, jo ka e^z ka map hai. Iske baad, hum y^2+1=0 ki solution ke liye, y coordinate kai jagahon par hoti hai, jaise 0, j, k, j+i, k+i aur so on. Iske liye, humein branch cuts ko chunna padta hai, jaise aisa koi line jo is plane ko chhor jaaye. 

Ab humein y^2+1=0 ki solution ko le jata hai phir iske baad humein y^(1/n)i ko y^(1/n)^(1/n) aur 0^(1/n) me convert karrta hai aur phir humein y^(1/n) = (-1)^(1/n) - 2^((n-1)/n) * e^(i*n*theta/n) me convert karta hai, aur phir theta = 2^(-n)*atan2(y^(1/n),

In [54]:
#next hamne ek chain banayi thi right
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
def formate_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text
parallel_chain = RunnableParallel({
    "context":retriever|RunnableLambda(formate_docs),
    "question":RunnablePassthrough()
})
parallel_chain.invoke("what is log")
parser=StrOutputParser()# ham stringoutputparser use kerenge right then
main_chain = parallel_chain|prompt|llm|parser
print(main_chain.invoke("can you samaraize this video "))

"Mujhe lagta hai ki yeh video ek mahaul se shuru hota hai, jo kai bade bade gyaanon ko cover karta hai. Lekin maine iske aaspaas ke mukhya bhaiyon ke baare mein baat rahi hai. Yeh video kuch naye tareekon se ek chitrakaar ke baare mein baat karti hai, jab unke nazar mein ek chitr me gaon dikhne lage. Yeh video 3B1B talent ko vichaar karta hai, jo ek aisa kaam hai jo logon ko apne paas samay kharch karne ke liye prerit karta hai. Is video ke shuruat mein hum ek gharane ki kalpana kartey hain, jahaan koi boat dikh reha hai, aur phir hum uss boat par chaltay hain jab main kaha krt hua ek village dikh rehta hai iska sab kuchh dhang se sahan karta hai.
